In [1]:
import pandas as pd

In [2]:
sd_f = pd.read_csv('model_data/student_data.csv')

In [3]:
sd_f.head()

,id_student,code_module,code_presentation,gender,highest_education,imd_band,age_band,disability,num_of_prev_attempts,studied_credits,site_count,CMA,TMA,Exam,final_result
0,11391,AAA,2013J,M,HE Qualification,90-100%,55<=,False,0,240,196.0,NaN,82.0,NaN,Pass
1,28400,AAA,2013J,F,HE Qualification,20-30%,35-55,False,0,60,430.0,NaN,66.4,NaN,Pass
2,30268,AAA,2013J,F,A Level or Equivalent,30-40%,35-55,True,0,60,76.0,NaN,NaN,NaN,Withdrawn
3,31604,AAA,2013J,F,A Level or Equivalent,50-60%,35-55,False,0,60,663.0,NaN,76.0,NaN,Pass
4,32885,AAA,2013J,F,Lower Than A Level,50-60%,0-35,False,0,60,352.0,NaN,54.4,NaN,Pass


In [4]:
sd_f.fillna({'site_count':0}, inplace=True)
sd_f.fillna({'CMA':0}, inplace=True)
sd_f.fillna({'TMA':0}, inplace=True)
sd_f.fillna({'Exam':0}, inplace=True)

,id_student,code_module,code_presentation,gender,highest_education,imd_band,age_band,disability,num_of_prev_attempts,studied_credits,site_count,CMA,TMA,Exam,final_result
0,11391,AAA,2013J,M,HE Qualification,90-100%,55<=,False,0,240,196.0,0.000000,82.000000,0.0,Pass
1,28400,AAA,2013J,F,HE Qualification,20-30%,35-55,False,0,60,430.0,0.000000,66.400000,0.0,Pass
2,30268,AAA,2013J,F,A Level or Equivalent,30-40%,35-55,True,0,60,76.0,0.000000,0.000000,0.0,Withdrawn
3,31604,AAA,2013J,F,A Level or Equivalent,50-60%,35-55,False,0,60,663.0,0.000000,76.000000,0.0,Pass
4,32885,AAA,2013J,F,Lower Than A Level,50-60%,0-35,False,0,60,352.0,0.000000,54.400000,0.0,Pass
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32588,2640965,GGG,2014J,F,Lower Than A Level,10-20%,0-35,False,0,30,19.0,0.000000,0.000000,0.0,Fail
32589,2645731,GGG,2014J,F,Lower Than A Level,40-50%,35-55,False,0,30,237.0,93.333333,77.666667,0.0,Distinction
32590,2648187,GGG,2014J,F,A Level or Equivalent,20-30%,0-35,True,0,30,108.0,80.000000,70.000000,0.0,Pass
32591,2679821,GGG,2014J,F,Lower Than A Level,90-100%,35-55,False,0,30,61.0,100.000000,83.000000,0.0,Withdrawn


In [5]:
sd_f['at_risk'] = sd_f['final_result'].apply(lambda x: 1 if x in ['Fail', 'Withdrawn'] else 0)

In [6]:
sd_f.drop(columns=['final_result', 'id_student'], inplace=True)

In [7]:
sd_f.info()

<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   code_module           32593 non-null  str    
 1   code_presentation     32593 non-null  str    
 2   gender                32593 non-null  str    
 3   highest_education     32593 non-null  str    
 4   imd_band              32593 non-null  str    
 5   age_band              32593 non-null  str    
 6   disability            32593 non-null  bool   
 7   num_of_prev_attempts  32593 non-null  int64  
 8   studied_credits       32593 non-null  int64  
 9   site_count            32593 non-null  float64
 10  CMA                   32593 non-null  float64
 11  TMA                   32593 non-null  float64
 12  Exam                  32593 non-null  float64
 13  at_risk               32593 non-null  int64  
dtypes: bool(1), float64(4), int64(3), str(6)
memory usage: 3.3 MB


In [8]:
category_cols = [
    'code_module',
    'code_presentation',
    'gender',
    'highest_education',
    'imd_band',
    'age_band',
    'disability'
]

sd_f = pd.get_dummies(sd_f, columns=category_cols, drop_first=True)

In [9]:
x_f = sd_f.drop(columns=['at_risk'])
y_f = sd_f['at_risk']

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
xf_train, xf_test, yf_train, yf_test = train_test_split(x_f, y_f, test_size=0.2, random_state=42)

In [12]:
from sklearn.preprocessing import StandardScaler

In [13]:
scaler_f = StandardScaler()

xf_train = scaler_f.fit_transform(xf_train)
xf_test = scaler_f.transform(xf_test)

In [14]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

In [15]:
param_grid_f = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs']
}

grid_f = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced'),
    param_grid_f,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

In [16]:
grid_f.fit(xf_train, yf_train)
model_final = grid_f.best_estimator_

c:\Users\Shalvi\anaconda3\envs\open_uni_prj\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [17]:
print("Best Params: ", grid_f.best_params_)

Best Params:  {'C': 10, 'penalty': 'l2', 'solver': 'lbfgs'}


In [18]:
yf_pred = model_final.predict(xf_test)

In [19]:
from sklearn.metrics import accuracy_score, classification_report

In [20]:
print('Accuracy:', accuracy_score(yf_test, yf_pred))

Accuracy: 0.8732934499156312


In [21]:
print(classification_report(yf_test, yf_pred))

              precision    recall  f1-score   support

           0       0.84      0.90      0.87      3051
           1       0.91      0.85      0.88      3468

    accuracy                           0.87      6519
   macro avg       0.87      0.87      0.87      6519
weighted avg       0.88      0.87      0.87      6519



In [22]:
feature_name_f = x_f.columns
coefficients_f = model_final.coef_[0]

coef_sd_f = pd.DataFrame({
    'Feature': feature_name_f,
    'Coefficient': coefficients_f
})

In [23]:
coef_sd_f.sort_values(by=['Coefficient'], ascending=False, inplace=True)

In [24]:
coef_sd_f

,Feature,Coefficient
7,code_module_CCC,2.096016
8,code_module_DDD,1.965472
10,code_module_FFF,1.164997
17,highest_education_Lower Than A Level,0.239749
9,code_module_EEE,0.234975
6,code_module_BBB,0.145405
1,studied_credits,0.133800
30,age_band_35-55,0.131088
31,age_band_55<=,0.111087
32,disability_True,0.084841


In [25]:
original_sd_f = pd.read_csv('model_data/student_data.csv')

original_sd_f.fillna({'site_count':0}, inplace=True)
original_sd_f.fillna({'CMA':0}, inplace=True)
original_sd_f.fillna({'TMA':0}, inplace=True)
original_sd_f.fillna({'Exam':0}, inplace=True)

,id_student,code_module,code_presentation,gender,highest_education,imd_band,age_band,disability,num_of_prev_attempts,studied_credits,site_count,CMA,TMA,Exam,final_result
0,11391,AAA,2013J,M,HE Qualification,90-100%,55<=,False,0,240,196.0,0.000000,82.000000,0.0,Pass
1,28400,AAA,2013J,F,HE Qualification,20-30%,35-55,False,0,60,430.0,0.000000,66.400000,0.0,Pass
2,30268,AAA,2013J,F,A Level or Equivalent,30-40%,35-55,True,0,60,76.0,0.000000,0.000000,0.0,Withdrawn
3,31604,AAA,2013J,F,A Level or Equivalent,50-60%,35-55,False,0,60,663.0,0.000000,76.000000,0.0,Pass
4,32885,AAA,2013J,F,Lower Than A Level,50-60%,0-35,False,0,60,352.0,0.000000,54.400000,0.0,Pass
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32588,2640965,GGG,2014J,F,Lower Than A Level,10-20%,0-35,False,0,30,19.0,0.000000,0.000000,0.0,Fail
32589,2645731,GGG,2014J,F,Lower Than A Level,40-50%,35-55,False,0,30,237.0,93.333333,77.666667,0.0,Distinction
32590,2648187,GGG,2014J,F,A Level or Equivalent,20-30%,0-35,True,0,30,108.0,80.000000,70.000000,0.0,Pass
32591,2679821,GGG,2014J,F,Lower Than A Level,90-100%,35-55,False,0,30,61.0,100.000000,83.000000,0.0,Withdrawn


In [26]:
db_sd = original_sd_f

x_predict = sd_f.drop(columns=['at_risk'])
x_f_scaled = scaler_f.transform(x_predict)

db_sd['predicted_risk'] = model_final.predict(x_f_scaled)

In [27]:
db_sd.head()

,id_student,code_module,code_presentation,gender,highest_education,imd_band,age_band,disability,num_of_prev_attempts,studied_credits,site_count,CMA,TMA,Exam,final_result,predicted_risk
0,11391,AAA,2013J,M,HE Qualification,90-100%,55<=,False,0,240,196.0,0.0,82.0,0.0,Pass,1
1,28400,AAA,2013J,F,HE Qualification,20-30%,35-55,False,0,60,430.0,0.0,66.4,0.0,Pass,0
2,30268,AAA,2013J,F,A Level or Equivalent,30-40%,35-55,True,0,60,76.0,0.0,0.0,0.0,Withdrawn,1
3,31604,AAA,2013J,F,A Level or Equivalent,50-60%,35-55,False,0,60,663.0,0.0,76.0,0.0,Pass,0
4,32885,AAA,2013J,F,Lower Than A Level,50-60%,0-35,False,0,60,352.0,0.0,54.4,0.0,Pass,0


In [28]:
db_sd.to_csv('model_data/student_dashboard_dataset.csv', index=False)